# Week 6 Lab — Working with real-world data

**HWRS 564a · Fall 2026**

Last week's CSV was already tidy, because I tidied it. This week you get data
the way it actually arrives: from an API, with gaps in it, duplicate rows, and
values that look impossible but aren't.

The skill this week is **deciding what to do about each of those**, and being
able to say why. There is no rule that covers it. There is domain knowledge and
there is a habit of looking.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Pull a record from the USGS API, with a fallback for when it isn't reachable
2. Tell the difference between missing, zero, and not-measured
3. Choose between dropping, filling, and interpolating — and defend the choice
4. Find and remove duplicate records
5. Flag statistical outliers, and decide which ones are real measurements
6. Write a cleaning function that reports what it changed

---

## Part 1 — Getting data from an API

An **API** is a URL that returns data instead of a web page. `dataretrieval`
wraps the USGS ones so you don't have to build the URLs yourself.

One thing to know before you start: **`dataretrieval` moved.** Tutorials you
find online will use `dataretrieval.nwis`, and much of that module is now
deprecated or gone — `nwis.get_gwlevels` no longer exists. The current API is
`dataretrieval.waterdata`. This is normal; libraries move, and the skill is
noticing when the tutorial you're reading is older than the library you have.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"

SABINO = "09484000"     # Sabino Creek near Tucson

### Always ship a fallback

Twelve people in one room hitting the same API at the same time is exactly when
it rate-limits you. A lab that depends on someone else's uptime is a lab that
fails in class, so the pattern is: **try live, fall back to a cached copy, and
say which one you got.**

In [ ]:
def load_sabino():
    """Daily discharge at Sabino Creek. Live if possible, cached if not."""
    try:
        import dataretrieval.nwis as nwis
        df = nwis.get_record(sites=SABINO, service="dv",
                             start="2005-01-01", end="2024-12-31")
        df = df.reset_index()
        df = pd.DataFrame({
            "datetime": pd.to_datetime(df["datetime"]).dt.tz_localize(None),
            "discharge_cfs": pd.to_numeric(df["00060_Mean"], errors="coerce"),
            "qualifier": df["00060_Mean_cd"],
        })
        print(f"live from NWIS: {len(df):,} rows")
        return df
    except Exception as exc:
        print(f"live fetch failed ({type(exc).__name__}); using the cached copy")
        return pd.read_csv(DATA / "cache" / f"nwis_{SABINO}_dv.csv",
                           parse_dates=["datetime"])


flow = load_sabino()
flow.head()

Either path gives you the same columns, so nothing downstream has to care which
one ran. That is the point of writing it as a function.

### YOUR TURN 1

Before you analyse anything, find out what you have. Compute:

- `n_days` — how many rows
- `n_missing` — how many rows have no discharge value
- `date_span_years` — years between the first and last date, as a float

In [ ]:
# YOUR TURN
n_days = ...
n_missing = ...
date_span_years = ...

In [ ]:
# CHECK
assert n_days == len(flow), f"expected {len(flow)}, got {n_days}"
assert n_missing == int(flow["discharge_cfs"].isna().sum())
assert abs(date_span_years - 19.99) < 0.1, f"expected about 20 years, got {date_span_years}"
print(f"{n_days:,} daily values over {date_span_years:.1f} years, "
      f"{n_missing} missing.")
print("Correct.")

---

## Part 2 — Missing, zero, and not-measured

These are three different things and conflating them is the most expensive
mistake in this notebook.

In [ ]:
zero_days = (flow["discharge_cfs"] == 0).sum()
nan_days = flow["discharge_cfs"].isna().sum()

print(f"days with discharge exactly 0.0 : {zero_days:,}")
print(f"days with no value at all       : {nan_days:,}")
print(f"fraction of the record at zero  : {zero_days / len(flow):.0%}")

**A third of the record is zero, and that is not an error.** Sabino Creek is
ephemeral — for much of the spring it genuinely has no flow. If you `dropna()`
your way to a "clean" dataset and then take a mean, you will report the mean of
the days it happened to be flowing and call it the mean flow.

The rule worth remembering: **`NaN` means "we don't know". `0` means "we know,
and it was zero".** Replacing one with the other destroys information in a way
you cannot get back.

In [ ]:
print(f"mean over the whole record   : {flow['discharge_cfs'].mean():8.2f} cfs")
print(f"mean of flowing days only    : {flow.loc[flow['discharge_cfs'] > 0, 'discharge_cfs'].mean():8.2f} cfs")
print(f"median over the whole record : {flow['discharge_cfs'].median():8.2f} cfs")

Three defensible numbers, three different questions. The median is zero, which
tells you something the means don't: on a typical day this creek is dry.

### The water-level record, which has real gaps

In [ ]:
levels = pd.read_csv(DATA / "tucson_water_levels.csv",
                     dtype={"site_no": str}, parse_dates=["date"])

one = levels[levels["site_no"] == "320824110593001"].sort_values("date").copy()
print(f"{len(one)} measurements, {one['date'].min():%Y} to {one['date'].max():%Y}")

gaps = one["date"].diff().dt.days
print(f"longest gap between readings: {gaps.max():.0f} days "
      f"({gaps.max() / 365:.1f} years)")
print(f"median gap: {gaps.median():.0f} days")

Manual water-level readings are irregular by nature — somebody drives out with a
tape. A multi-year gap is not missing data to be filled; it is a period nobody
measured.

### YOUR TURN 2

Three strategies for the same gappy series. Build all three so you can compare
them, using a small example where you can see every value.

- `dropped` — rows with a missing value removed
- `filled` — missing values replaced with the **column mean**
- `interpolated` — missing values **linearly interpolated** between neighbours

In [ ]:
demo = pd.Series([36.5, np.nan, 41.2, np.nan, np.nan, 52.8, 55.1])
print(demo.tolist())

# YOUR TURN
dropped = ...
filled = ...
interpolated = ...

In [ ]:
# CHECK
assert len(dropped) == 4, f"expected 4 rows left, got {len(dropped)}"
assert abs(filled.iloc[1] - demo.mean()) < 1e-9, "filled should use the column mean"
assert abs(interpolated.iloc[1] - 38.85) < 1e-9, f"got {interpolated.iloc[1]}"
assert abs(interpolated.iloc[3] - 45.07) < 0.01, f"got {interpolated.iloc[3]}"
print(f"dropped      : {dropped.round(2).tolist()}")
print(f"filled       : {filled.round(2).tolist()}")
print(f"interpolated : {interpolated.round(2).tolist()}")
print("Correct.")

**Now the part that matters.** Look at what each one did:

- `dropped` is honest but shortens the series, and if you later plot it against
  an unmodified time axis the points land in the wrong places.
- `filled` put 46.4 — roughly the series average — between 36.5 and 41.2. That
  is a *rise* that never happened. Mean-filling a trending series invents
  reversals.
- `interpolated` put 38.85, halfway between its neighbours. For a slowly-varying
  quantity measured irregularly, this is usually right.

For a water table that moves smoothly, interpolate. For daily streamflow with a
week missing during a monsoon, interpolating draws a straight line through a
flood that may or may not have happened — there, leave the gap.

**The method depends on the physics, not on the shape of the DataFrame.**

---

## Part 3 — Duplicates

Repeated rows come from re-running a download, from a station reporting twice,
or from merging two overlapping exports. They quietly weight some observations
double.

In [ ]:
dupes = levels.duplicated(subset=["site_no", "date"], keep=False)
print(f"{dupes.sum()} rows share a site and date with another row")

if dupes.sum():
    print(levels[dupes].sort_values(["site_no", "date"]).head(6).to_string(index=False))

Note the `subset=` argument. Two readings of the same well on the same day are
suspicious even if the *values* differ — which is exactly the case a plain
`.duplicated()` with no subset would miss, because it only flags rows that match
in every column.

In [ ]:
# A deliberately dirty copy so there is something to find
dirty = pd.concat([one, one.head(3)], ignore_index=True)
print(f"dirty: {len(dirty)} rows")
print(f"exact duplicates:            {dirty.duplicated().sum()}")
print(f"duplicate site+date pairs:   {dirty.duplicated(subset=['site_no', 'date']).sum()}")

clean = dirty.drop_duplicates(subset=["site_no", "date"], keep="first")
print(f"after drop_duplicates:       {len(clean)} rows")

---

## Part 4 — Outliers, and which ones are real

A statistical outlier is a value far from the others. Whether it is an *error*
is a separate question, and statistics cannot answer it.

The standard tool is the **interquartile range**: flag anything more than 1.5
IQRs beyond the first or third quartile.

In [ ]:
q = flow["discharge_cfs"]

q1, q3 = q.quantile(0.25), q.quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f}   Q3 = {q3:.2f}   IQR = {iqr:.2f}")
print(f"IQR fence: [{low:.2f}, {high:.2f}]")
print(f"flagged as outliers: {((q < low) | (q > high)).sum():,} of {len(q):,} days "
      f"({((q < low) | (q > high)).mean():.0%})")

**Twenty percent of the record is an "outlier".** The method has not failed —
it was designed for roughly symmetric data, and streamflow is nothing of the
sort. On a creek that is dry half the year, any flow at all is far from the
median.

### YOUR TURN 3

Find the largest daily flow in the record and the date it happened.

- `peak_cfs` — the maximum daily mean discharge
- `peak_date` — the date it occurred, as a Timestamp
- `peak_month` — the month number it fell in

In [ ]:
# YOUR TURN
peak_cfs = ...
peak_date = ...
peak_month = ...

In [ ]:
# CHECK
assert abs(peak_cfs - 2450.0) < 1.0, f"expected about 2450 cfs, got {peak_cfs}"
assert peak_month in (7, 8), f"expected a monsoon month, got month {peak_month}"
print(f"largest daily mean flow: {peak_cfs:,.0f} cfs on {peak_date:%Y-%m-%d}")
print(f"that is {peak_cfs / q[q > 0].median():,.0f}x the median flowing-day discharge")
print("Correct.")

That peak is roughly two thousand times the typical flowing-day discharge, and
**it is completely real** — a monsoon thunderstorm over the Santa Catalinas. Drop
it as an outlier and you have removed the single most important day in the
record for anyone sizing a culvert or estimating recharge.

Compare that with a value that is genuinely impossible:

In [ ]:
suspicious = pd.Series([31.2, 45.8, -999.0, 52.1, 48.9])

print("IQR flags:", suspicious[(suspicious < suspicious.quantile(0.25) - 1.5 *
      (suspicious.quantile(0.75) - suspicious.quantile(0.25)))].tolist())
print("physics flags:", suspicious[suspicious < 0].tolist())

`-999` is a sentinel value — a convention from fixed-width data formats for
"no data", which pandas has no way to recognise. **A depth to water cannot be
negative.** That test comes from knowing what the number means, not from its
distribution.

Both checks are worth running. The statistical one finds things to *look at*;
the physical one finds things to *remove*.

### YOUR TURN 4

Write `flag_impossible(series, minimum, maximum)` returning a boolean Series
that is `True` where the value is outside the physically possible range **or**
is one of the common sentinel values `-999` or `-9999`.

In [ ]:
# YOUR TURN
def flag_impossible(series, minimum, maximum):
    """True where a value is physically impossible or a known sentinel."""
    ...

In [ ]:
# CHECK
test = pd.Series([31.2, -999.0, 45.8, 1500.0, -9999.0, 0.0, 52.1])
flags = flag_impossible(test, minimum=0.0, maximum=1200.0)

assert flags.dtype == bool, f"should return booleans, got {flags.dtype}"
assert flags.tolist() == [False, True, False, True, True, False, False], flags.tolist()
assert not flags.iloc[5], "zero is a legitimate depth to water — don't flag it"
print(f"flagged {flags.sum()} of {len(test)} values: {test[flags].tolist()}")
print("Correct.")

---

## Part 5 — A cleaning function that tells you what it did

The worst cleaning code is the kind that silently drops half your data. Make it
report.

In [ ]:
def clean_levels(df, min_depth=0.0, max_depth=1500.0, verbose=True):
    """Remove duplicates and impossible values from a water-level table.

    Returns the cleaned frame. Prints what it removed, because a cleaning step
    you can't audit is a cleaning step you can't trust.
    """
    n0 = len(df)
    out = df.drop_duplicates(subset=["site_no", "date"], keep="first")
    n_dup = n0 - len(out)

    bad = flag_impossible(out["depth_to_water_ft"], min_depth, max_depth)
    out = out[~bad]

    if verbose:
        print(f"  {n0:,} rows in")
        print(f"  -{n_dup:,} duplicate site/date pairs")
        print(f"  -{bad.sum():,} physically impossible values")
        print(f"  {len(out):,} rows out ({len(out) / n0:.1%} kept)")
    return out


cleaned = clean_levels(levels)

### YOUR TURN 5

Apply the whole pipeline to the Sabino Creek record and answer one question
about it: in which **month** does the creek most often run dry?

Build `dry_fraction_by_month` — a Series indexed by month number 1–12, giving
the fraction of days in that month with exactly zero flow.

In [ ]:
# YOUR TURN
dry_fraction_by_month = ...

In [ ]:
# CHECK
assert len(dry_fraction_by_month) == 12, f"expected 12 months, got {len(dry_fraction_by_month)}"
assert dry_fraction_by_month.idxmax() in (5, 6), \
    f"expected May or June to be driest, got month {dry_fraction_by_month.idxmax()}"
assert dry_fraction_by_month.max() > 0.6, "the driest month should be dry most days"
print(dry_fraction_by_month.round(2).to_string())
print(f"\ndriest month: {dry_fraction_by_month.idxmax()}")
print("Correct.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(dry_fraction_by_month.index, dry_fraction_by_month * 100,
       color="#AB0520", alpha=0.85)
ax.set_xticks(range(1, 13))
ax.set_xlabel("month")
ax.set_ylabel("% of days with zero flow")
ax.set_title("Sabino Creek runs dry in the pre-monsoon foresummer")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

**Think about this before next week:** the dry season is May–June and the creek
recovers in July. That is the North American monsoon arriving, and it is the
single strongest signal in Arizona hydrology.

Next week you will pull that signal out properly, with resampling and rolling
windows, instead of by grouping on a month number.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

**HW 4 — pandas fundamentals**, Wednesday 9/30 at 11:59pm, through D2L.

## Next week

Time series properly: `DatetimeIndex`, `resample`, `groupby`, and rolling
windows.

## Stuck?

- If the live API call hangs, interrupt the kernel (■). The fallback path is
  there precisely so you can carry on without it.
- `SettingWithCopyWarning` when you assign after filtering means you need a
  `.copy()` on the filtered frame.
- A `.mean()` that returns `NaN` means every value in the column is missing;
  one that looks wrong usually means some rows were skipped. `.isna().sum()`
  and `.count()` tell you which.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.